# OMOP CDM Gold Layer Overview

This notebook provides an overview of the Gold layer that transforms FHIR Silver tables into OMOP Common Data Model (CDM) v5.4 format.

## Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  SILVER LAYER (FHIR Resources)                                              │
│  Patient | Encounter | Condition | Claim | Coverage | Procedure | ...       │
└────────────────────────────────┬────────────────────────────────────────────┘
                                 │ Streaming Transformations
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│  GOLD LAYER (OMOP CDM v5.4)                                                 │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Clinical Data Tables                                                │   │
│  │  • person                    (from Patient)                          │   │
│  │  • visit_occurrence          (from Encounter)                        │   │
│  │  • condition_occurrence      (from Condition)                        │   │
│  │  • drug_exposure             (from MedicationRequest/Administration) │   │
│  │  • procedure_occurrence      (from Procedure/ServiceRequest)         │   │
│  │  • measurement               (from Observation - labs/vitals)        │   │
│  │  • observation               (from Observation - other)              │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Health System Tables                                                │   │
│  │  • care_site                 (from Organization/Location)            │   │
│  │  • provider                  (from Practitioner)                     │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Vocabulary Tables (Reference)                                       │   │
│  │  • concept                   (OMOP Vocabulary)                       │   │
│  │  • concept_relationship      (Vocabulary mappings)                   │   │
│  │  • vocabulary                (Vocabulary metadata)                   │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────────┘
```

## FHIR to OMOP Mapping Summary

| FHIR Resource | OMOP Table | Key Mappings |
|---------------|------------|---------------|
| Patient | person | gender→gender_concept_id, birthDate→birth_datetime |
| Encounter | visit_occurrence | class→visit_concept_id, period→dates |
| Condition | condition_occurrence | code→condition_concept_id (SNOMED/ICD) |
| MedicationRequest | drug_exposure | medication→drug_concept_id (RxNorm) |
| Procedure | procedure_occurrence | code→procedure_concept_id (SNOMED/CPT) |
| Observation (lab) | measurement | code→measurement_concept_id (LOINC) |
| Observation (other) | observation | code→observation_concept_id |
| Practitioner | provider | identifier→provider_id |
| Organization | care_site | identifier→care_site_id |

## Vocabulary Mapping

The OMOP CDM uses standardized concept IDs. Common mappings:

| Source Vocabulary | OMOP Standard | Use Case |
|-------------------|---------------|----------|
| SNOMED CT | Standard | Conditions, Procedures |
| ICD-10-CM | Non-standard → SNOMED | Conditions |
| RxNorm | Standard | Medications |
| LOINC | Standard | Lab tests, Vitals |
| CPT-4 | Non-standard → SNOMED | Procedures |
| HCPCS | Non-standard → SNOMED | Procedures |

## References

- [OMOP CDM v5.4 Documentation](https://ohdsi.github.io/CommonDataModel/cdm54.html)
- [HL7 FHIR to OMOP IG](https://build.fhir.org/ig/HL7/fhir-omop-ig/)
- [OHDSI Vocabulary](https://athena.ohdsi.org/)

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';  -- Silver tables location
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';      -- OMOP CDM tables location
DECLARE OR REPLACE VARIABLE vocab_schema STRING DEFAULT 'vocabulary'; -- OMOP vocabulary tables

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);
SET VARIABLE vocab_schema = COALESCE(:vocab_schema, vocab_schema);

SELECT 
  catalog_use AS catalog,
  silver_schema AS fhir_silver,
  gold_schema AS omop_gold,
  vocab_schema AS omop_vocabulary;

In [ ]:
-- Create Gold schema if not exists
DECLARE OR REPLACE VARIABLE create_schema_stmt STRING;
SET VARIABLE create_schema_stmt = 'CREATE SCHEMA IF NOT EXISTS ' || catalog_use || '.' || gold_schema;
EXECUTE IMMEDIATE create_schema_stmt;

In [ ]:
-- Verify Silver tables are available
USE IDENTIFIER(catalog_use || '.' || silver_schema);

SELECT 
  table_name,
  table_type
FROM information_schema.tables
WHERE table_schema = silver_schema
ORDER BY table_name;

## OMOP Standard Concept IDs Reference

Common concept IDs used in transformations:

### Gender Concepts
| Gender | concept_id |
|--------|------------|
| Male | 8507 |
| Female | 8532 |
| Unknown | 0 |

### Visit Concepts
| Visit Type | concept_id |
|------------|------------|
| Inpatient Visit | 9201 |
| Outpatient Visit | 9202 |
| Emergency Room Visit | 9203 |
| Long Term Care Visit | 42898160 |
| ER + Inpatient Visit | 262 |

### Type Concepts
| Type | concept_id | Use Case |
|------|------------|----------|
| EHR | 32817 | condition_type_concept_id |
| Prescription written | 38000177 | drug_type_concept_id |
| EHR procedure | 32817 | procedure_type_concept_id |
| Lab result | 32856 | measurement_type_concept_id |
| EHR observation | 32817 | observation_type_concept_id |